# 01 - BBOB Problem Noise Landscape Analysis

This notebook focuses on **BBOB problem creation**, **additive Gaussian noise injection**, **landscape topology visualization**, and **empirical noise distribution analysis** across multiple standard deviations (`noise_std = [0.0, 0.2, 0.5, 0.8, 1.0, 2.0, 5.0]`) using interactive Plotly graphics.

In [124]:
# Setup paths and imports
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# # Ensure local src directory is on sys.path
# src_dir = Path.cwd()
# while src_dir.name != "AAD_LLM" and src_dir.parent != src_dir:
#     src_dir = src_dir.parent
# src_path = str(src_dir / "src")
# if src_path not in sys.path:
#     sys.path.insert(0, src_path)

import aad_llm.problems.bbob as bbob_module
import importlib
importlib.reload(bbob_module)
from aad_llm.problems.bbob import BBOBProblem


In [125]:
problem_id = 14
dim = 2
instance_id = 1
noise_levels = [0.0, 0.2, 0.5, 0.8, 1.0, 2.0, 5.0]

problem = BBOBProblem(problem_id=problem_id, dim=dim, instance_id=instance_id)

print(f"Loaded BBOB Problem {problem_id} ({dim}D)")
print(f"Domain Bounds: [{problem.lower_bound}, {problem.upper_bound}] (vector lb={problem.lb}, ub={problem.ub})")
print(f"True target optimum (global minimum): {problem.true_optimum:.4f}")


Loaded BBOB Problem 14 (2D)
Domain Bounds: [-5.0, 5.0] (vector lb=[-5. -5.], ub=[5. 5.])
True target optimum (global minimum): -52.3500


In [126]:
np.random.seed(42)
lb, ub = problem.lower_bound, problem.upper_bound
points = np.random.uniform(lb, ub, (100, dim))

clean_y = np.array([problem(x) for x in points])
sort_idx = np.argsort(clean_y)
sorted_clean = clean_y[sort_idx]
indices = np.arange(len(points))

# Combined Plot showing all noise std levels overlaid on clean ground truth
fig_combined = go.Figure()
fig_combined.add_trace(go.Scatter(
    x=indices,
    y=sorted_clean,
    mode='lines',
    name='Clean Fitness (Ground Truth)',
    line=dict(color='#111827', width=3.5)
))

color_palette = {
    0.2: '#10B981',  # emerald green
    0.5: '#3B82F6',  # blue
    0.8: '#8B5CF6',  # purple
    1.0: '#F59E0B',  # amber
    2.0: '#EC4899',  # pink
    5.0: '#EF4444'   # red
}

for std in [0.2, 0.5, 0.8, 1.0, 2.0, 5.0]:
    noisy_y = np.array([problem.add_noise(y, std) for y in clean_y])
    sorted_noisy = noisy_y[sort_idx]
    fig_combined.add_trace(go.Scatter(
        x=indices,
        y=sorted_noisy,
        mode='markers',
        name=f'std = {std}',
        marker=dict(color=color_palette.get(std, '#6B7280'), size=5, opacity=0.75)
    ))

fig_combined.update_layout(
    title=f"BBOB Problem {problem_id}: Clean vs. Noisy Fitness across Noise Levels [0.2, 0.5, 0.8, 1.0, 2.0, 5.0]",
    xaxis_title="Sample Points (Sorted by Clean Fitness)",
    yaxis_title="Fitness Value",
    template="plotly_white",
    hovermode='x unified',
    height=600,
    width=1000
)
fig_combined.show()

# Subplot Grid for side-by-side noise inspection (2 rows x 3 cols)
noisy_stds = [0.2, 0.5, 0.8, 1.0, 2.0, 5.0]
fig_subplots = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'noise_std = {std}' for std in noisy_stds]
)

coords = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]
for idx, (std, (r, c)) in enumerate(zip(noisy_stds, coords), start=1):
    noisy_y = np.array([problem.add_noise(y, std) for y in clean_y])
    sorted_noisy = noisy_y[sort_idx]
    
    # Clean reference line
    fig_subplots.add_trace(
        go.Scatter(x=indices, y=sorted_clean, mode='lines', name='Clean', line=dict(color='#111827', width=2), showlegend=(idx==1)),
        row=r, col=c
    )
    # Noisy markers
    fig_subplots.add_trace(
        go.Scatter(x=indices, y=sorted_noisy, mode='markers', name=f'std={std}', marker=dict(color=color_palette.get(std, '#EF4444'), size=5, opacity=0.75), showlegend=(idx==1)),
        row=r, col=c
    )

fig_subplots.update_layout(
    title_text=f"BBOB Problem {problem_id}: Side-by-Side Noise Standard Deviation Grid",
    height=650,
    width=1050,
    template="plotly_white"
)
fig_subplots.show()


In [127]:
lb, ub = problem.lower_bound, problem.upper_bound
x_range = np.linspace(lb, ub, 100)
y_range = np.linspace(lb, ub, 100)
X, Y = np.meshgrid(x_range, y_range)

landscapes = {}
for std in noise_levels:
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            pt = np.array([X[i, j], Y[i, j]])
            if std == 0.0:
                Z[i, j] = problem(pt)
            else:
                res = problem(pt, noise_std=std)
                Z[i, j] = res[std]
    landscapes[std] = Z

contour_stds = [0.0, 0.2, 0.8, 1.0, 2.0, 5.0]
fig_contour = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'BBOB f{problem_id} (noise_std = {std})' for std in contour_stds]
)

coords = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]
for std, (r, c) in zip(contour_stds, coords):
    fig_contour.add_trace(
        go.Contour(x=x_range, y=y_range, z=np.log10(np.maximum(landscapes[std] - problem.true_optimum, 1e-8)), colorscale='Viridis', showscale=(r==1 and c==3)),
        row=r, col=c
    )

fig_contour.update_layout(title_text="2D Contour Comparison (Log10 Scale) across Noise Levels", height=700, width=1050)
fig_contour.show()


In [128]:
all_stds = [0.0, 0.2, 0.5, 0.8, 1.0, 2.0, 5.0]

fig_3d_titles = [f"noise_std = {std} ({'Clean' if std==0.0 else 'Noisy'})" for std in all_stds]
fig_3d = make_subplots(
    rows=2, cols=4,
    specs=[[{'type': 'surface'} for _ in range(4)] for _ in range(2)],
    subplot_titles=fig_3d_titles
)

coords_3d = [(1, 1), (1, 2), (1, 3), (1, 4), (2, 1), (2, 2), (2, 3)]
colorscales_3d = ['Viridis', 'Cividis', 'Blues', 'Purples', 'Oranges', 'Plasma', 'Inferno']

for std, (r, c), cscale in zip(all_stds, coords_3d, colorscales_3d):
    # Plot original surface for 3D topology without log10 scaling
    fig_3d.add_trace(
        go.Surface(
            x=X, y=Y, 
            z=landscapes[std], 
            colorscale=cscale, 
            showscale=False
        ),
        row=r, col=c
    )

fig_3d.update_layout(
    title_text=f"3D Surface Topography Comparison<br><sup>{problem.full_meta_str} | bounds = [{problem.lower_bound}, {problem.upper_bound}]</sup>",
    height=800,
    width=1200,
    margin=dict(l=20, r=20, t=80, b=20)
)


In [129]:
n_samples = 1000
test_pt = np.zeros(dim)

fig = go.Figure()
for std in [0.2, 0.5, 0.8, 1.0, 2.0, 5.0]:
    samples = [problem(test_pt, noise_std=std)[std] for _ in range(n_samples)]
    fig.add_trace(go.Histogram(
        x=samples,
        opacity=0.45,
        name=f'std={std} (emp std={np.std(samples):.2f})',
        marker_color=color_palette.get(std, '#6B7280'),
        nbinsx=30
    ))

true_val = problem(test_pt)
fig.add_vline(x=true_val, line_dash="dash", line_color="black", annotation_text="True Optimum Value")

fig.update_layout(
    title_text="Empirical Noise Distribution across 1000 Evaluations at Origin for all Noise Levels",
    xaxis_title="Evaluated Fitness Value",
    yaxis_title="Frequency",
    barmode='overlay',
    height=500,
    width=1000
)
fig.show()
